# 11 · Ampliación opcional. Escalado robusto

Comparamos la misma regresión lineal con y sin escalado robusto. El objetivo es ver qué cambia en la representación de las variables y comprobar si cambia el error.

## 0 · Preparar el entorno

Elige un kernel de **Python 3.11 o 3.12** con **Java 17**. En Jupyter o VS Code usa el mismo entorno que en las prácticas anteriores. En Colab, comprueba la versión con `!python --version`. Si muestra Python 3.13, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** selecciona la versión **2026.07**, que usa Python 3.12. Si no tienes Java 17, ejecuta en una celda de Colab `!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless` y, en otra celda, `import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"`.

Si faltan las bibliotecas, ejecuta en una celda `%pip install "pyspark>=3.5,<3.6" "numpy<2"` y reinicia el kernel si te lo pide. Puedes consultar la [guía del curso](../01_environment_pyspark.md). Ejecuta las celdas de arriba abajo. Los datos de ejemplo están dentro del notebook y el cálculo se hace en el equipo donde corre el kernel.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import RobustScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder.master('local[2]').appName('Escalado robusto')
         .config('spark.sql.shuffle.partitions', '4').getOrCreate())
print(f'Spark {spark.version}')

## 1 · Crear y revisar la tabla de ejemplo

Una fila representa a una persona ficticia. Las horas de formación previa y la jornada semanal se conocen antes de intentar predecir un resultado. El salario mensual y la finalización del curso son dos resultados diferentes. Usamos una semilla para generar la misma tabla en los cuadernos 11, 12 y 14.

La encuesta es **simulada**. Sirve para practicar el método y sus métricas.

In [ ]:
from random import Random

azar = Random(42)
filas = []

for persona in range(1, 361):
    # Tres perfiles de partida hacen visible la idea de segmentación.
    experiencia_base, jornada_base, formacion_base = azar.choice([
        (3, 24, 12), (8, 35, 28), (18, 40, 8),
    ])
    experiencia = max(0, experiencia_base + azar.randint(-3, 3))
    jornada = jornada_base + azar.randint(-4, 4)
    formacion = max(0, formacion_base + azar.randint(-8, 8))
    salario = round(650 + 60 * experiencia + 23 * jornada + azar.gauss(0, 220), 2)
    # La formación previa ayuda y una jornada más larga deja menos tiempo.
    puntuacion = formacion - 0.45 * jornada + azar.gauss(0, 9)
    completa = int(puntuacion >= 1)
    filas.append((persona, experiencia, jornada, formacion, salario, completa))

datos = spark.createDataFrame(filas, [
    'persona', 'anios_experiencia', 'horas_jornada', 'horas_formacion',
    'salario_mensual', 'completa_curso',
])

print(f'Filas: {datos.count()}')
datos.show(5)

## 2 · Elegir lo que queremos predecir

La variable objetivo es `salario_mensual`. La experiencia y las horas semanales de jornada son las dos entradas. `VectorAssembler` coloca las entradas de cada fila en la columna `variables`, en ese mismo orden. Spark espera un vector para entrenar el modelo.

In [ ]:
entradas = ['anios_experiencia', 'horas_jornada']
ensamblador = VectorAssembler(inputCols=entradas, outputCol='variables')
preparados = ensamblador.transform(datos)
preparados.select('salario_mensual', 'variables').show(5)

## 3 · Separar entrenamiento y prueba

El modelo aprende solo con **entrenamiento**. Reservamos **prueba** para simular lo que ocurre cuando llegan filas nuevas. El reparto es aproximadamente 80/20. La semilla permite repetir la división con los mismos datos.

In [ ]:
entrenamiento, prueba = preparados.randomSplit([0.8, 0.2], seed=42)
print(f'Entrenamiento: {entrenamiento.count()} · Prueba: {prueba.count()}')

## 4 · Ajustar el escalado con entrenamiento

El escalado robusto cambia las unidades usando la mediana y el rango intercuartílico, la distancia entre los percentiles 25 y 75. Ajustamos el escalador con entrenamiento y aplicamos esa misma transformación en prueba. Así evitamos usar información de la prueba durante la preparación.

In [ ]:
escalador = RobustScaler(inputCol='variables', outputCol='variables_escaladas',
                         withCentering=True, withScaling=True).fit(entrenamiento)
entrenamiento_escalado = escalador.transform(entrenamiento)
prueba_escalada = escalador.transform(prueba)
prueba_escalada.select('variables', 'variables_escaladas').show(5, truncate=False)

## 5 · Comparar con las mismas filas de prueba

Ajustamos dos regresiones sin penalización. Solo cambia la escala de entrada. Evaluamos el MAE en euros y comparamos los resultados.

In [ ]:
def evaluar_mae(datos_entrenamiento, datos_prueba, columna):
    modelo = LinearRegression(featuresCol=columna, labelCol='salario_mensual').fit(datos_entrenamiento)
    predicciones = modelo.transform(datos_prueba)
    evaluador = RegressionEvaluator(labelCol='salario_mensual', metricName='mae')
    return evaluador.evaluate(predicciones)

print(f'Sin escalar: {evaluar_mae(entrenamiento, prueba, "variables"):.2f} €')
print(f'Con escalado robusto: {evaluar_mae(entrenamiento_escalado, prueba_escalada, "variables_escaladas"):.2f} €')

**Interpretación.** Una regresión lineal sin penalización puede producir prácticamente las mismas predicciones después de este cambio de escala. Los coeficientes pasan a expresarse en otras unidades. En K-Means veremos un caso donde la escala sí cambia las distancias entre observaciones.

## Cerrar la sesión

Liberamos los recursos de Spark al terminar este cuaderno.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Regresión lineal guiada](11_census_linear_regression.ipynb) · [Siguiente: Clasificación con regresión logística](12_census_logistic_regression.ipynb)

Documentación de Spark ML. [Regresión y clasificación](https://spark.apache.org/docs/3.5.6/ml-classification-regression.html) · [Preparación de variables](https://spark.apache.org/docs/3.5.6/ml-features.html)